# 数组与数据处理

学习目标：能用数组组织数据，选择变换、筛选、汇总与排序方法，并解释稀疏和共享引用的影响。

前置知识：函数与回调、对象属性、解构和展开、循环、严格相等与浅拷贝。

适用版本：ECMAScript 2025（ECMA-262 第 16 版）、Node.js 24.11.0；.mjs 文件按 ES 模块运行并采用严格模式。console 是宿主输出 API。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/10-arrays-and-data-processing/。

1. [main.mjs](scripts/10-arrays-and-data-processing/main.mjs)：按正文顺序运行全部正常示例。
2. [empty-reduce.mjs](scripts/10-arrays-and-data-processing/empty-reduce.mjs)：空数组 reduce 没有初始值，也没有可用作起点的第一个元素。
3. [with-out-of-range.mjs](scripts/10-arrays-and-data-processing/with-out-of-range.mjs)：with 只替换现有范围内的位置，不负责扩展数组。
4. [invalid-length.mjs](scripts/10-arrays-and-data-processing/invalid-length.mjs)：Array 的 length 不能设置为负数或小数。

Step 1：从项目根目录进入本章工作目录。

```bash
cd content/编程语言/javascript
```

Step 2：运行全部正常示例，按各片段中的输出注释核对。

```bash
node scripts/10-arrays-and-data-processing/main.mjs
```

下文正常片段依次对应 main.mjs 中的代码；每段给出自身输入与定义。错误文件仅在相应小节单独运行。

## 1 数组、索引与 length

Array 是对象类型中的序列容器，索引属性从 0 开始。length 是合法范围为 0–2 ** 32 - 1 的整数，但并不等于实际存在的元素数量；缺失索引会形成空槽（hole）。数组索引最大为 2 ** 32 - 2，其他属性名仍可作为普通对象属性，却不参与这套长度更新规则。

new Array(3) 创建长度为 3 的空槽数组，Array.of(3) 创建只含元素 3 的数组；通常用字面量避免单个数字参数的歧义。Array.from() 可从可迭代值或类数组创建数组，并可接收转换函数。类数组指具有 length 和相应索引访问的值，不因此自动成为 Array。

增大 length 不创建对应元素；缩小 length 会从末尾删除索引属性。本例的 length 可写、待删除索引均可配置；若遇到不可配置索引，截断会中止，严格模式下抛出 TypeError，已经删除的尾项不会恢复。

at(-1) 从末尾取值，方括号 [-1] 则是名为 "-1" 的属性。

```javascript
const values = [10, 20];
console.log(Array.isArray(values), values[0], values.at(-1), values[-1]);
values[4] = 50;
console.log(values.length, Object.hasOwn(values, 2), values[2]);
values.length = 2;
console.log(values.length, values[4]);
console.log(new Array(3).length, Object.hasOwn(new Array(3), 0), Array.of(3).join(","));
console.log(Array.from("A🚀").length);
console.log(Array.from({ length: 3 }, (value, index) => index + 1).join(","));
// 输出依次为：
// true 10 20 undefined
// 5 false undefined
// 2 undefined
// 3 false 3
// 2
// 1,2,3
```

## 2 增删改与原地方法

这些方法直接修改原数组；返回值不一定是数组。本表 items 表示待加入的元素，start 为起始索引，deleteCount 为删除数量。

| 完整名称 | 中文名称／含义 | 返回值 |
| --- | --- | --- |
| Array.prototype.push(...items) | 从末尾添加 | 新 length |
| Array.prototype.pop() | 删除末尾元素 | 被删元素，空数组为 undefined |
| Array.prototype.unshift(...items) | 从开头添加 | 新 length |
| Array.prototype.shift() | 删除开头元素 | 被删元素，空数组为 undefined |
| Array.prototype.splice(start, deleteCount, ...items) | 在指定位置删除及插入 | 被删元素组成的新数组 |

splice() 的负起点从末尾计算，省略删除数量时从起点删到结尾。delete 只删除索引属性并留下空槽，不移动后面的元素，也不缩短 length，不能替代 splice()。

```javascript
const queue = ["B"];
console.log(queue.push("C"), queue.unshift("A"), queue.join(","));
console.log(queue.shift(), queue.pop(), queue.join(","));
const removed = queue.splice(0, 1, "X", "Y");
console.log(removed.join(","), queue.join(","));
queue[1] = "Z";
delete queue[0];
console.log(queue.length, Object.hasOwn(queue, 0), queue[1]);
console.log([].pop(), [].shift());
// 输出依次为：
// 2 3 A,B,C
// A C B
// B X,Y
// 2 false Z
// undefined undefined
```

## 3 查找值和查找符合条件的元素

indexOf() 按严格相等查找，返回索引或 -1；includes() 返回布尔值，采用 SameValueZero 比较，因此能找到 NaN，同时将正负零视为相等。两者对对象都比较身份。

find() 返回第一个使回调结果为真值的元素，没有命中则为 undefined；findIndex() 返回对应索引或 -1。元素本来就是 undefined 时，只看 find() 的返回值无法区分命中和未命中，需要索引或其他显式判断。findLast()、findLastIndex() 从末尾开始找。下面 item 是当前元素，回调中的条件决定是否命中。

```javascript
const numbers = [4, 8, NaN, 8];
console.log(numbers.indexOf(8), numbers.indexOf(7), numbers.includes(NaN), numbers.indexOf(NaN));
console.log(numbers.find(item => item > 5), numbers.findIndex(item => item > 5));
console.log(numbers.findLast(item => item === 8), numbers.findLastIndex(item => item === 8));
console.log(numbers.find(item => item > 99), numbers.findIndex(item => item > 99));
const item = { id: 1 };
console.log([item].includes(item), [item].includes({ id: 1 }));
// 输出依次为：
// 1 -1 true -1
// 8 1
// 8 3
// undefined -1
// true false
```

## 4 切片、解构和展开

slice(start, end) 返回起点到终点之前的浅拷贝，支持负索引，省略终点则到末尾，不修改原数组。concat() 返回连接后的新数组，对普通数组参数展开一层；数组展开也能创建新数组，但都不会递归复制元素对象。

数组解构按迭代顺序取值，逗号可跳过一个位置，默认值只在值为 undefined 时生效；...tail 收集剩余项，必须放最后。下面 first、third、tail 是接收结果的变量。

```javascript
const letters = ["A", "B", "C", "D"];
console.log(letters.slice(1, 3).join(","), letters.slice(-2).join(","), letters.join(","));
const [first, , third, ...tail] = letters;
console.log(first, third, tail.join(","));
const [fallback = "缺省"] = [];
console.log(fallback);
const appended = [...letters, "E"];
console.log(appended.join(","), letters.concat(["E"]).join(","));
// 输出依次为：
// B,C C,D A,B,C,D
// A C D
// 缺省
// A,B,C,D,E A,B,C,D,E
```

## 5 map、filter 与回调返回值

map() 将每次回调的返回值放在结果数组对应位置；filter() 根据回调结果的真值决定是否保留原元素。它们本身不直接修改输入数组，但回调仍可能修改输入。常见回调参数依次为当前元素、索引、原数组，不能把一个参数含义不同的现成函数直接交进去。

带花括号的箭头函数若忘记 return，会返回 undefined。map() 不会因为返回 undefined 就删掉该元素；filter() 则会把 undefined 当作假值。需要副作用时可用 forEach()，它忽略回调返回值并返回 undefined，不能用回调里的 return 提前结束整个遍历。

下面 parseInt 本来把第二个实参当作进制；直接作为 map 回调时，索引却占据这个位置。用箭头明确只传入文本并固定进制，能让接口匹配。

```javascript
const prices = [2, 5, 8];
console.log(prices.map(value => value * 2).join(","));
console.log(prices.filter(value => value >= 5).join(","), prices.join(","));
const missingReturn = prices.map(value => { value * 2; });
console.log(missingReturn.length, missingReturn[0], Object.hasOwn(missingReturn, 0));
console.log(prices.filter(value => { value >= 5; }).length);
const decimalText = ["10", "10", "10"];
console.log(decimalText.map(Number.parseInt).join(","));
console.log(decimalText.map(text => Number.parseInt(text, 10)).join(","));
let visited = 0;
const foreachResult = prices.forEach(value => { visited += 1; return value; });
console.log(visited, foreachResult);
// 输出依次为：
// 4,10,16
// 5,8 2,5,8
// 3 undefined true
// 0
// 10,NaN,2
// 10,10,10
// 3 undefined
```

## 6 reduce、some 与 every

reduce() 按顺序累积一个结果。回调参数依次是累积值、当前元素、索引和原数组，必须返回下一轮要用的累积值。本例 sum 和 value 分别表示累计和与当前数值。提供初始值 0 后，空数组也能返回 0；未提供初始值时，以第一个存在的元素作为起点，没有任何可用元素则抛出 TypeError。

some() 在某项满足条件时立即返回 true，every() 在某项不满足时立即返回 false，后续回调不再调用。空数组的 some() 是 false，every() 是 true；后者仅表示没有反例，不能代替“非空且全部通过”的业务判断。

```javascript
const scores = [3, 6, 9];
console.log(scores.reduce((sum, value) => sum + value, 0), [].reduce((sum, value) => sum + value, 0));
let checks = 0;
const hasLarge = scores.some(value => { checks += 1; return value >= 6; });
console.log(hasLarge, checks, scores.every(value => value > 0));
console.log([].some(value => value > 0), [].every(value => value > 0));
console.log(scores.length > 0 && scores.every(value => value > 0));
// 输出依次为：
// 18 0
// true 2 true
// false true
// true
```

## 7 扁平化与 flatMap

flat(depth) 将嵌套数组摊平指定层数，depth 默认是 1，Infinity 表示持续摊平遇到的数组层次；这仍不复制数组里对象的内部内容。它只摊平真正的数组，不把任意可迭代对象自动拆开。

flatMap() 对每个元素调用转换函数，再把返回数组摊平一层。返回空数组可删除该项，返回多个元素可扩展该项；返回非数组值就保留该值本身。它不会自动递归摊平任意层数。

```javascript
const nested = [1, [2, [3]]];
const oneLevel = nested.flat();
console.log(oneLevel.length, Array.isArray(oneLevel[2]), nested.flat(Infinity).join(","));
const expanded = [1, 2, 3].flatMap(value => value % 2 === 0 ? [] : [value, value * 10]);
console.log(expanded.join(","));
console.log(["AB"].flatMap(value => value).join(","));
// 输出依次为：
// 3 true 1,2,3
// 1,10,3,30
// AB
```

## 8 按键分组

Object.groupBy(items, callback) 从可迭代的 items 收集分组，callback 接收当前元素和索引，返回分组键；键会转换为字符串或 Symbol。它是 Object 的静态方法，不是 Array 实例方法。结果是无原型对象，每个属性值是保留相应元素的新数组。

各组元素仍引用输入中的原对象，分组不会复制记录。需要对象等任意值本身作为键时，可用 Map.groupBy()，其集合接口在下一章展开。空输入得到没有分组属性的对象。

```javascript
const products = [
  { name: "笔", type: "文具" },
  { name: "杯", type: "生活" },
  { name: "纸", type: "文具" },
];
const groups = Object.groupBy(products, product => product.type);
console.log(Object.keys(groups).join(","), groups["文具"].length);
console.log(groups["文具"].map(product => product.name).join(","));
console.log(Object.getPrototypeOf(groups) === null, groups["文具"][0] === products[0]);
console.log(Object.keys(Object.groupBy([], value => value)).length);
// 输出依次为：
// 文具,生活 2
// 笔,纸
// true true
// 0
```

## 9 排序规则与稳定性

sort() 原地排序并返回原数组。省略比较函数时，除 undefined 等特殊处理外，会把元素转为字符串，按 UTF-16 码元顺序排序，因此数值数组通常需要比较函数。

比较函数接收两个待比较元素，返回负 Number 表示前者排在前，正 Number 表示排在后，0 表示同组；不是返回布尔值。对于本例的有限 Number，(left, right) => left - right 表示升序。比较器应对相同输入给出一致结果；交换参数后正负号应相反，相等时仍为 0，并满足传递性，不能随意随机或边比较边修改数据。

ECMAScript 2025 要求稳定排序：比较结果为 0 的元素保留原先相对顺序。具体比较次数和内部算法不属于这里保证的输出。toSorted() 使用相同的比较规则，但返回新数组，适合保留输入顺序。

```javascript
const sourceNumbers = [10, 2, 1];
console.log(sourceNumbers.toSorted().join(","));
const ordered = sourceNumbers.toSorted((left, right) => left - right);
console.log(ordered.join(","), sourceNumbers.join(","));
console.log(sourceNumbers.sort((left, right) => left - right) === sourceNumbers, sourceNumbers.join(","));
const records = [{ id: "A", rank: 2 }, { id: "B", rank: 1 }, { id: "C", rank: 2 }];
console.log(records.toSorted((left, right) => left.rank - right.rank).map(record => record.id).join(","));
// 输出依次为：
// 1,10,2
// 1,2,10 10,2,1
// true 1,2,10
// B,A,C
```

## 10 复制型修改方法

复制型方法为改变顺序、增删或替换创建新数组，仍只做浅层复制。下面 index 是要替换的索引，value 是新值；负索引从末尾计数。

| 完整名称 | 中文名称／含义 | 原数组与返回值 |
| --- | --- | --- |
| Array.prototype.reverse() | 原地反转 | 改原数组，返回原数组 |
| Array.prototype.toReversed() | 反转副本 | 保留原数组，返回新数组 |
| Array.prototype.toSorted(comparator) | 排序副本 | 保留原数组，返回新数组 |
| Array.prototype.toSpliced(start, skipCount, ...items) | 在副本中删除及插入 | 保留原数组，返回变更后的新数组 |
| Array.prototype.with(index, value) | 在副本中替换一个索引 | 保留原数组，返回新数组 |

toSpliced() 返回变更后的完整数组，不能把它与 splice() 返回被删元素数组混淆。with() 的索引必须落在现有长度范围内，超界会抛出 RangeError，不会像普通索引赋值那样扩展数组。

```javascript
const original = [1, 2, 3];
console.log(original.toReversed().join(","));
console.log(original.toSpliced(1, 1, 8, 9).join(","));
console.log(original.with(-1, 7).join(","), original.join(","));
const mutable = [1, 2, 3];
console.log(mutable.reverse() === mutable, mutable.join(","));
// 输出依次为：
// 3,2,1
// 1,8,9,3
// 1,2,7 1,2,3
// true 3,2,1
```

## 11 稀疏数组不是 undefined 数组

空槽表示没有该索引属性，显式保存 undefined 表示属性存在。以下例子假设原型链没有同名索引属性；普通属性查找会包括原型，不能把数组方法的所有操作都简化为只检查自有元素。

map() 跳过不存在的索引并在结果中保留空槽；filter()、reduce()、some()、every()、forEach() 也不对不存在的索引调用回调。find() 系列则读取范围内的每个索引，空槽通常表现为 undefined。includes() 同样可把空槽当作 undefined 查到，indexOf() 则跳过空槽。

slice() 保留空槽；展开、Array.from() 和 toReversed()、toSorted()、toSpliced()、with() 会在结果中为相应位置创建值，通常将保留下来的空槽变为显式 undefined。flat() 会跳过其处理范围中的空槽。不要仅靠 join() 的显示结果判断属性是否存在。

```javascript
const sparse = [, 2, undefined];
console.log(sparse.length, Object.hasOwn(sparse, 0), Object.hasOwn(sparse, 2));
let mapCalls = 0;
const mapped = sparse.map(value => { mapCalls += 1; return value; });
console.log(mapCalls, Object.hasOwn(mapped, 0), Object.hasOwn(mapped, 2));
console.log(sparse.filter(() => true).length);
let findCalls = 0;
sparse.find(value => { findCalls += 1; return false; });
console.log(findCalls);
console.log([,].includes(undefined), [,].indexOf(undefined));
const sliced = sparse.slice();
const spread = [...sparse];
console.log(Object.hasOwn(sliced, 0), Object.hasOwn(spread, 0), Object.hasOwn(Array.from(sparse), 0));
console.log(Object.hasOwn(sparse.toReversed(), 2), [1, , [2, , 3]].flat().join(","));
// 输出依次为：
// 3 false true
// 2 false true
// 2
// 3
// true -1
// false true true
// true 1,2,3
```

## 12 共享引用与回调中的修改

新数组不等于深拷贝。slice()、展开、filter()、复制型方法以及分组结果里的对象元素，通常仍引用原来的对象。要隔离对象属性变化，需按数据结构复制相应层级；任意复杂值还要考虑上一章的克隆限制。

fill(object) 会把同一个对象值放到多个位置，不会为每个位置调用构造逻辑。需要每项独立时可使用 Array.from() 的回调逐项创建。

map() 等回调方法在开始时确定访问长度，但读取某个索引时使用当时的值；追加的尾项不会因此加入原定范围，尚未访问项的修改可能被看到。因此不要一边依赖输入不变，一边在回调中修改同一数组。下面有意用小例子观察这种时机，实际变换优先保持输入稳定。

```javascript
// 先比较只复制外层数组和同时复制对象元素，观察共享引用在哪一层。
const sourceRows = [{ count: 1 }];
const shallow = sourceRows.slice();
shallow[0].count = 4;
console.log(shallow !== sourceRows, shallow[0] === sourceRows[0], sourceRows[0].count);
const separate = sourceRows.map(row => ({ ...row }));
separate[0].count = 8;
console.log(sourceRows[0].count, separate[0].count);
// fill 重复放入同一个对象；Array.from 的回调每次创建新对象。
const repeated = new Array(2).fill({ count: 0 });
repeated[0].count = 1;
const independent = Array.from({ length: 2 }, () => ({ count: 0 }));
independent[0].count = 1;
console.log(repeated[1].count, independent[1].count);
// 有意在回调中改输入：第二项的新值会被读取，新追加的第三项不在原访问范围。
const changing = [1, 2];
const observed = changing.map((value, index) => {
  if (index === 0) {
    changing[1] = 9;
    changing.push(3);
  }
  return value;
});
console.log(observed.join(","), changing.join(","));
// 输出依次为：
// true true 4
// 4 8
// 1 0
// 1,9 1,9,3
```

## 13 独立观察错误边界

以下文件分别启动新进程；预期退出码为 1。先根据代码判断错误原因，再运行相应命令，核对错误名称及对应位置。错误消息全文由宿主决定。

空数组 reduce 没有初始值，也没有可用作起点的第一个元素。

```javascript
console.log([].reduce((sum, value) => sum + value));
// 预期错误：TypeError；Reduce of empty array with no initial value
```

Step 1：独立运行 scripts/10-arrays-and-data-processing/empty-reduce.mjs。

```bash
node scripts/10-arrays-and-data-processing/empty-reduce.mjs
```

with 只替换现有范围内的位置，不负责扩展数组。

```javascript
console.log([1, 2].with(2, 3));
// 预期错误：RangeError；Invalid index
```

Step 2：独立运行 scripts/10-arrays-and-data-processing/with-out-of-range.mjs。

```bash
node scripts/10-arrays-and-data-processing/with-out-of-range.mjs
```

Array 的 length 不能设置为负数或小数。

```javascript
const values = [];
values.length = 1.5;
// 预期错误：RangeError；Invalid array length
```

Step 3：独立运行 scripts/10-arrays-and-data-processing/invalid-length.mjs。

```bash
node scripts/10-arrays-and-data-processing/invalid-length.mjs
```

## 本章小结

- 选方法时先判断是否修改原数组，再判断返回的是元素、长度、汇总值还是新数组。
- 回调的参数、返回值与提前结束规则必须匹配；空输入需要明确定义。
- 排序需提供正确比较规则，分组需选择适合的键类型。
- 空槽与 undefined 不同，新数组与独立对象也不同。

## 练习

1. 对 [8, 3, 10, 1] 筛选不小于 3 的数，乘以 2 后求和；核对为 42，并确认原数组仍为 8,3,10,1。
2. 按产品 type 分组，再统计各组数量；核对文具为 2、生活为 1，空输入时没有分组键。
3. 对 rank 相同的多条记录排序；核对同组 id 的原有相对顺序不变，并比较 sort() 和 toSorted() 对原数组的影响。
4. 创建长度为 3、仅索引 1 有值的数组；用 Object.hasOwn() 核对 slice() 保留空槽，而展开后所有索引都存在。
5. 创建三个互不共享的计数对象，仅修改第一个；核对后两个 count 保持 0。修复 empty-reduce 时必须给出空输入的合理初始值。

## 参考与引用来源

- TC39（tc39.es）：[§23.1 Array：构造、方法与 length](https://tc39.es/ecma262/2025/multipage/indexed-collections.html#sec-array-objects)、[§10.4.2.4 ArraySetLength：截断与失败处理](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-arraysetlength)、[§23.1.3.21 map](https://tc39.es/ecma262/2025/multipage/indexed-collections.html#sec-array.prototype.map)、[§23.1.3.8 filter](https://tc39.es/ecma262/2025/multipage/indexed-collections.html#sec-array.prototype.filter)、[§23.1.3.24 reduce](https://tc39.es/ecma262/2025/multipage/indexed-collections.html#sec-array.prototype.reduce)、[§23.1.3.30 稳定排序及比较器](https://tc39.es/ecma262/2025/multipage/indexed-collections.html#sec-array.prototype.sort)、[§23.1.3.33–23.1.3.39 复制型方法](https://tc39.es/ecma262/2025/multipage/indexed-collections.html#sec-array.prototype.toreversed)、[§20.1.2.13 Object.groupBy](https://tc39.es/ecma262/2025/multipage/fundamental-objects.html#sec-object.groupby)、[§7.3.35 分组键转换](https://tc39.es/ecma262/2025/multipage/abstract-operations.html#sec-groupby)、[§13.15.5 解构赋值](https://tc39.es/ecma262/2025/multipage/ecmascript-language-expressions.html#sec-destructuring-assignment)、[§13.2.4 数组字面量与展开](https://tc39.es/ecma262/2025/multipage/ecmascript-language-expressions.html#sec-array-initializer)、[§24.1.2.1 Map.groupBy 的键类型](https://tc39.es/ecma262/2025/multipage/keyed-collections.html#sec-map.groupby)：ECMAScript 2025 的数组方法、回调、空槽、顺序、分组和复制规则。
- MDN：[Indexed collections](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Indexed_collections)：数组创建、操作、稀疏数组及数据处理用法对照。